# Pelatihan Model & Evaluasi: DUCFF
Di sini kita memuat data preprocessed dan melatih model DUCFF raksasa.

In [ ]:
import numpy as np
import pandas as pd
import time
import pickle
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Bidirectional, Dropout, LayerNormalization
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load preprocessed data
data = np.load('processed_data.npz')
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
TIME_STEPS = X_train.shape[1]
input_shape = (TIME_STEPS, X_train.shape[2])

with open('rps_scaler.pkl', 'rb') as f:
    rps_scaler = pickle.load(f)


## Arsitektur DUCFF

In [ ]:
inputs = Input(shape=input_shape)
x_tcn = tf.keras.layers.Conv1D(filters=128, kernel_size=2, padding='causal', activation='relu')(inputs)
x_tcn = tf.keras.layers.Conv1D(filters=64, kernel_size=2, padding='causal', activation='relu')(x_tcn)
x_tcn_flat = tf.keras.layers.Flatten()(x_tcn)
attention_output = tf.keras.layers.MultiHeadAttention(num_heads=8, key_dim=64)(inputs, inputs)
attention_output = LayerNormalization(epsilon=1e-6)(attention_output + inputs)
x_transformer_flat = tf.keras.layers.Flatten()(attention_output)
fusion = tf.keras.layers.Concatenate()([x_tcn_flat, x_transformer_flat])
x = Dense(256, activation='relu')(fusion)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
outputs = Dense(1, activation='linear')(x)
model = Model(inputs=inputs, outputs=outputs, name="DUCFF_Monster")
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
csv_logger = CSVLogger('training_progress_DUCFF.csv', append=True)
EPOCHS = 50
BATCH_SIZE = 64

print(f"\n--- Training: DUCFF ---")
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, csv_logger], verbose=1)


## Evaluasi Kecepatan & Akurasi

In [ ]:
single_sample = X_test[0:1]

# Pemanasan prediksi
_ = model.predict(single_sample, verbose=0)

# Precise inference benchmark
t_arr = []
for _ in range(30):
    t_s = time.time()
    model.predict(single_sample, verbose=0)
    t_arr.append(time.time() - t_s)
avg_inference_ms = np.mean(t_arr) * 1000

# Prediksi seluruh data test
pred = model.predict(X_test, verbose=0)
np.save('pred_DUCFF.npy', pred)

results = {
    'DUCFF': {
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE': mean_absolute_error(y_test, pred),
        'R_Squared (R²)': r2_score(y_test, pred),
        'Speed (ms)': avg_inference_ms
    }
}
display(pd.DataFrame(results).T)
